# Env utils

> Fill in a module description here

In [ ]:
#| default_exp utils.env_utils

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations

from typing import Any, Callable

import gymnasium as gym
import numpy as np
import torch
from gymnasium.vector.utils import batch_space



In [ ]:
#| export
"""EnvPool: vectorized env wrapper with selective stepping.

A lightweight replacement for ``gymnasium.vector.SyncVectorEnv`` with two
differences tailored to ``World``:

1. ``reset(mask=...)`` and ``step(mask=...)`` can skip individual envs —
   useful for the ``wait`` reset mode where terminated envs freeze until
   every env has finished.
2. The stacked info dict is pre-allocated on the first reset and updated
   in-place afterwards. Tensor/array values are shaped ``(num_envs, 1, ...)``
   so consumers can rely on a ``(batch, time, ...)`` convention without the
   pool re-stacking every step.
"""

class EnvPool:
    """Batched env runner with selective stepping.

    Args:
        env_fns: List of zero-arg factories, one per env. Each is called
            once and the result is kept for the lifetime of the pool.
    """

    def __init__(self, env_fns: list):
        self.envs = [fn() for fn in env_fns]
        self._single_env = self.envs[0]
        self._stacked_infos: dict[str, Any] | None = None
        self.seeds = np.zeros(len(self.envs), dtype=np.int64)
        # Cache batched spaces — rebuilding them per-access creates a fresh
        # unseeded space each call, so .seed() / .sample() never advances RNG.
        self._action_space = batch_space(
            self._single_env.action_space, len(self.envs)
        )
        self._observation_space = batch_space(
            self._single_env.observation_space, len(self.envs)
        )

    @property
    def num_envs(self) -> int:
        """Number of envs in the pool."""
        return len(self.envs)

    @property
    def action_space(self) -> gym.Space:
        """Batched action space (``batch_space(single_action_space, num_envs)``)."""
        return self._action_space

    @property
    def single_action_space(self) -> gym.Space:
        """Action space of a single env."""
        return self._single_env.action_space

    @property
    def observation_space(self) -> gym.Space:
        """Batched observation space."""
        return self._observation_space

    @property
    def single_observation_space(self) -> gym.Space:
        """Observation space of a single env."""
        return self._single_env.observation_space

    @property
    def variation_space(self):
        """Variation space from the unwrapped env, or ``None`` if not defined."""
        return getattr(self._single_env.unwrapped, 'variation_space', None)

    @property
    def single_variation_space(self):
        """Variation space for a single env (alias of ``variation_space``)."""
        return self.variation_space

    def reset(
        self,
        seed: int | list[int | None] | None = None,
        options: dict | list[dict | None] | None = None,
        mask: np.ndarray | None = None,
    ) -> tuple[None, dict]:
        """Reset envs and return the stacked info dict.

        Args:
            seed: Base int (each env gets ``seed + i``), a per-env list,
                or ``None``.
            options: Shared dict or per-env list.
            mask: If provided, only envs where ``mask[i]`` is truthy are
                reset. Others keep their current state in the stacked
                info buffer.
        """
        seeds = _broadcast_arg(seed, self.num_envs, increment=True)
        opts = _broadcast_arg(options, self.num_envs)

        per_env_infos = [None] * self.num_envs
        for i, env in enumerate(self.envs):
            if mask is not None and not mask[i]:
                continue
            _, per_env_infos[i] = env.reset(seed=seeds[i], options=opts[i])
            if seeds[i] is not None:
                self.seeds[i] = seeds[i]

        if self._stacked_infos is None or mask is None:
            self._stacked_infos = _stack_fresh(per_env_infos)
        else:
            for i, info in enumerate(per_env_infos):
                if info is not None:
                    _write_env_info(self._stacked_infos, i, info)

        return None, self._stacked_infos

    def step(
        self, actions: np.ndarray, mask: np.ndarray | None = None
    ) -> tuple[None, np.ndarray, np.ndarray, np.ndarray, dict]:
        """Step envs and return ``(None, rewards, terminateds, truncateds, infos)``.

        Args:
            actions: Array of shape ``(num_envs, ...)`` — one action per env.
            mask: If provided, only envs where ``mask[i]`` is truthy are
                stepped. Masked envs contribute zero reward and ``False``
                termination/truncation, and their slot in the stacked
                info buffer is left unchanged.
        """
        rewards = np.zeros(self.num_envs, dtype=np.float32)
        terminateds = np.zeros(self.num_envs, dtype=bool)
        truncateds = np.zeros(self.num_envs, dtype=bool)

        for i, env in enumerate(self.envs):
            if mask is not None and not mask[i]:
                continue
            _, rewards[i], terminateds[i], truncateds[i], info = env.step(
                actions[i]
            )
            _write_env_info(self._stacked_infos, i, info)

        return None, rewards, terminateds, truncateds, self._stacked_infos

    def close(self):
        """Close every env in the pool."""
        for env in self.envs:
            env.close()


def _broadcast_arg(arg, n: int, increment: bool = False) -> list:
    if arg is None:
        return [None] * n
    if isinstance(arg, list):
        return arg
    if isinstance(arg, np.ndarray):
        return list(arg)
    if increment and isinstance(arg, int):
        return [arg + i for i in range(n)]
    return [arg] * n


def _stack_fresh(per_env_infos: list[dict]) -> dict[str, Any]:
    """Build stacked info arrays from a full set of per-env infos.

    Tensor/array values get a leading time dim of 1 after the env dim,
    yielding shape (N, 1, ...) so downstream consumers can rely on a
    (batch, time, ...) convention.
    """
    keys = per_env_infos[0].keys()
    stacked = {}
    for k in keys:
        vals = [info[k] for info in per_env_infos]
        first = vals[0]
        if isinstance(first, torch.Tensor):
            stacked[k] = torch.stack(vals).unsqueeze(1)
        elif isinstance(first, np.ndarray):
            stacked[k] = np.stack(vals)[:, None, ...]
        elif isinstance(first, (bool, int, float, np.number)):
            stacked[k] = np.array(vals)[:, None]
        else:
            stacked[k] = [[v] for v in vals]
    return stacked


def _write_env_info(stacked: dict, idx: int, info: dict) -> None:
    """Write a single env's info into pre-allocated stacked arrays in-place."""
    for k, v in info.items():
        if k not in stacked:
            continue
        buf = stacked[k]
        if isinstance(buf, torch.Tensor):
            if not isinstance(v, torch.Tensor):
                v = torch.as_tensor(v, dtype=buf.dtype, device=buf.device)
            buf[idx, 0] = v
        elif isinstance(buf, np.ndarray):
            buf[idx, 0] = v
        elif isinstance(buf, list):
            buf[idx][0] = v

In [ ]:
#| export
"""Multi-agent env pool over N independent multi-agent envs.

Unlike an earlier "agents as slots in one shared env" design, this treats
each PettingZooWrapper instance as a complete, independent multi-agent env --
exactly EnvPool's actual pattern (a list of env instances), except each "env"
here is itself multi-agent. `num_envs` is the number of parallel EPISODES,
not the number of agents; each agent's data is stacked across those N
episodes, exactly like `info_dict["pixels"]` etc. are stacked across
`num_envs` in the single-agent `World`/`EnvPool` case.

Consequences of this design (vs. the shared-env version):
  - `mask` in reset()/step() means exactly what it means in `EnvPool`: skip
    that env entirely. No no-op substitution hack needed.
  - Because each env's `agents` property shrinks as individual agents
    terminate (independently per env), `step()` must build each env's action
    dict from that env's *currently live* agents, not from the pool's fixed
    `agents` list. `possible_agents` (fixed, per env) is what's used for
    stable per-agent stacking across the whole run.
  - Termination is naturally per (agent, env) pair: `terminateds[agent][i]`.
"""


class MultiAgentEnvPool:
    """Pool of N independent multi-agent (PettingZoo ParallelEnv) envs.

    Args:
        env_fns: List of zero-arg factories, one per parallel episode. Each
            is called once (mirrors `EnvPool`) and kept for the pool's
            lifetime.
        agents: Fixed list of agent ids used for per-agent stacking (should
            equal each env's `possible_agents`; assumed identical/homogeneous
            across envs, as EnvPool assumes for its single envs).
    """

    def __init__(self, env_fns: list[Callable[[], Any]], agents: list | None = None):
        self.envs = [fn() for fn in env_fns]
        self._single_env = self.envs[0]
        self.agents = list(agents) if agents is not None else list(self._single_env.possible_agents)

        self._stacked_infos: dict[Any, dict[str, Any]] | None = None
        self.seeds = np.zeros(len(self.envs), dtype=np.int64)

        self._action_spaces = {
            a: batch_space(self._single_env.action_space(a), self.num_envs)
            for a in self.agents
        }
        self._observation_spaces = {
            a: batch_space(self._single_env.observation_space(a), self.num_envs)
            for a in self.agents
        }

    @property
    def num_envs(self) -> int:
        """Number of parallel episodes (NOT number of agents)."""
        return len(self.envs)

    def action_space(self, agent):
        return self._action_spaces[agent]

    def observation_space(self, agent):
        return self._observation_spaces[agent]

    def single_action_space(self, agent):
        return self._single_env.action_space(agent)

    def single_observation_space(self, agent):
        return self._single_env.observation_space(agent)

    def reset(
        self,
        seed: int | list[int | None] | None = None,
        options: dict | list[dict | None] | None = None,
        mask: np.ndarray | None = None,
    ) -> tuple[None, dict]:
        """Reset envs and return per-agent stacked info dicts.

        Returns:
            (None, stacked_infos) where stacked_infos is
            ``{agent_id: {key: array of shape (num_envs, 1, ...)}}``.
        """
        seeds = _broadcast_arg(seed, self.num_envs, increment=True)
        opts = _broadcast_arg(options, self.num_envs)

        per_agent_infos = {a: [None] * self.num_envs for a in self.agents}
        for i, env in enumerate(self.envs):
            if mask is not None and not mask[i]:
                continue
            obs, infos = env.reset(seed=seeds[i], options=opts[i])
            if seeds[i] is not None:
                self.seeds[i] = seeds[i]
            for a in self.agents:
                if a in obs:
                    per_agent_infos[a][i] = {**obs[a], **infos.get(a, {})}

        if self._stacked_infos is None or mask is None:
            self._stacked_infos = {
                a: _stack_fresh([info or {} for info in per_agent_infos[a]])
                for a in self.agents
            }
        else:
            for a in self.agents:
                for i, info in enumerate(per_agent_infos[a]):
                    if info is not None:
                        _write_env_info(self._stacked_infos[a], i, info)

        return None, self._stacked_infos

    def step(
        self, actions: dict[Any, np.ndarray], mask: np.ndarray | None = None,
        noop_action: Any = 3,
    ) -> tuple[None, dict, dict, dict, dict]:
        """Step envs and return per-agent (rewards, terminateds, truncateds, infos).

        Args:
            actions: ``{agent_id: array of shape (num_envs, ...)}``.
            mask: If provided, envs where ``mask[i]`` is falsy are skipped
                entirely (their stacked info stays unchanged) -- same
                meaning as in `EnvPool`.
            noop_action: Action substituted for any agent no longer in that
                env's live-agent set (`env.agents`). We always supply an
                entry for every `possible_agent`, not just live ones --
                this codebase's `MultiGridEnv.step()` appears to expect an
                action for every possible agent each call (mirrors the
                original evaluator, which always filled done agents with a
                no-op rather than omitting them from the actions dict).

        Returns:
            (None, rewards, terminateds, truncateds, stacked_infos), each of
            the first four a dict ``{agent_id: array of shape (num_envs,)}``.
        """
        rewards = {a: np.zeros(self.num_envs, dtype=np.float32) for a in self.agents}
        terminateds = {a: np.zeros(self.num_envs, dtype=bool) for a in self.agents}
        truncateds = {a: np.zeros(self.num_envs, dtype=bool) for a in self.agents}

        for i, env in enumerate(self.envs):
            if mask is not None and not mask[i]:
                continue

            live_agents = set(env.agents)  # dynamic: shrinks as agents terminate
            step_actions = {
                a: (actions[a][i] if a in live_agents else noop_action)
                for a in self.agents if a in actions
            }

            obs, r, term, trunc, infos = env.step(step_actions)

            for a in self.agents:
                if a not in obs:
                    continue
                rewards[a][i] = r.get(a, 0.0)
                terminateds[a][i] = term.get(a, False)
                truncateds[a][i] = trunc.get(a, False)
                info = {**obs[a], **infos.get(a, {})}
                _write_env_info(self._stacked_infos[a], i, info)

        return None, rewards, terminateds, truncateds, self._stacked_infos

    def close(self) -> None:
        for env in self.envs:
            env.close()


def set_env_state(pool, env_idx, goal_pos, agent_positions, agent_directions, agent_ids):
    """Directly overwrite one env's state to match a recorded episode config.

    Args:
        pool: MultiAgentEnvPool.
        env_idx: which env instance (i.e. which parallel episode slot).
        agent_positions: dict[agent_id -> (x, y)] for this env only.
        agent_directions: dict[agent_id -> int] for this env only.
        agent_ids: agent ids to overwrite in this env.
    """
    unwrapped = pool.envs[env_idx].env.unwrapped
    goal_pos = tuple(int(v) for v in goal_pos)

    old_goal_pos = tuple(unwrapped.goal_pos)
    if old_goal_pos != goal_pos:
        goal_obj = unwrapped.grid.get(*old_goal_pos)
        unwrapped.grid.set(*old_goal_pos, None)
        unwrapped.grid.set(*goal_pos, goal_obj)
    unwrapped.goal_pos = goal_pos

    for agent_id in agent_ids:
        agent = unwrapped.agents[agent_id]
        agent.state.pos = tuple(int(v) for v in agent_positions[agent_id])
        agent.state.dir = int(agent_directions[agent_id])

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()